# Project 88 — Local Audio Transcription Summary
## Transcript → Structured Notes → Action Items

**Stack:** LangChain · Ollama · Pydantic · Jupyter

*Uses pre-existing transcript text to demonstrate the summary pipeline.*

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Sample Meeting Transcripts

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

transcripts = [
    {
        "title": "Q1 Sprint Retrospective",
        "duration": "45 min",
        "text": """
Sarah: Good morning everyone. Let's start with what went well this sprint.
Mike: The new API integration shipped ahead of schedule. Team collaboration was great.
Alice: Agreed, the pair programming sessions really helped. Code review turnaround dropped from 2 days to 4 hours.
Sarah: Great. What didn't go well?
Bob: The deployment pipeline broke twice. We need better monitoring.
Mike: Also, the requirements for the dashboard feature changed three times mid-sprint.
Sarah: Let's talk about action items.
Alice: I'll set up PagerDuty alerts for the pipeline. Due by Friday.
Bob: I'll draft a change request process document. Should have it by next Wednesday.
Sarah: Mike, can you lead the requirements workshop with product?
Mike: Sure, I'll schedule it for next Monday.
Sarah: Perfect. Any other concerns?
Bob: We should probably upgrade our test infrastructure. The CI builds are taking 45 minutes now.
Sarah: Good point. Let's add that to the next sprint backlog. Meeting adjourned."""
    },
    {
        "title": "Product Strategy Review",
        "duration": "30 min",
        "text": """
CEO: Let's review our Q2 product strategy. Revenue target is $5M.
VP Product: We have three main initiatives: AI features, enterprise tier, and mobile app.
CEO: What's the AI features timeline?
VP Eng: Beta in April, GA in May. We need two more ML engineers.
VP Product: The enterprise tier is our biggest revenue opportunity — $2M projected.
CEO: What's blocking enterprise?
VP Product: SOC2 compliance. We're about 60% through the audit.
VP Eng: Should be complete by end of March if we prioritize it.
CEO: Make it top priority. What about mobile?
VP Product: MVP is on track for June. We're using React Native for cross-platform.
CEO: Good. Hiring — VP Eng, how many do we need?
VP Eng: Two ML engineers, one DevOps, and one security engineer. Budget impact: $600K annually.
CEO: Approved. Let's reconvene in two weeks with progress updates."""
    },
]
print(f"Transcripts to process: {len(transcripts)}")

## Step 2 — Structured Meeting Notes

In [ ]:
class ActionItem(BaseModel):
    owner: str
    task: str
    deadline: str
    priority: str = Field(description="high, medium, low")

class MeetingNotes(BaseModel):
    title: str
    participants: list[str]
    duration: str
    summary: str = Field(description="2-3 sentence overview")
    key_decisions: list[str]
    action_items: list[ActionItem]
    open_questions: list[str]
    risks: list[str]
    next_steps: str

note_extractor = llm.with_structured_output(MeetingNotes)

meeting_notes = []
for transcript in transcripts:
    notes = note_extractor.invoke(
        f"Extract structured meeting notes:\n"
        f"Title: {transcript['title']}\n"
        f"Duration: {transcript['duration']}\n\n"
        f"Transcript:\n{transcript['text']}"
    )
    meeting_notes.append(notes)
    print(f"\n{'='*50}")
    print(f"📋 {notes.title}")
    print(f"Participants: {', '.join(notes.participants)}")
    print(f"Summary: {notes.summary}")
    print(f"\nDecisions: {len(notes.key_decisions)}")
    for d in notes.key_decisions:
        print(f"  • {d}")
    print(f"\nAction Items: {len(notes.action_items)}")
    for a in notes.action_items:
        print(f"  [{a.priority}] {a.owner}: {a.task} (due: {a.deadline})")

## Step 3 — Follow-Up Email Draft

In [ ]:
email_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a professional follow-up email summarizing the meeting. "
     "Include action items with owners and deadlines. Keep it concise."),
    ("human", "Meeting: {title}\nNotes: {notes}\n\nEmail:")
])
email_chain = email_prompt | llm | StrOutputParser()

for notes in meeting_notes:
    email = email_chain.invoke({
        "title": notes.title,
        "notes": json.dumps(notes.model_dump(), indent=2),
    })
    print(f"\n--- Email for '{notes.title}' ---")
    print(email[:400])
    print("...")

## Step 4 — Action Item Tracker

In [ ]:
import pandas as pd

all_actions = []
for notes in meeting_notes:
    for a in notes.action_items:
        all_actions.append({
            "meeting": notes.title[:25],
            "owner": a.owner,
            "task": a.task[:40],
            "deadline": a.deadline,
            "priority": a.priority,
            "status": "pending",
        })

df = pd.DataFrame(all_actions)
print("ACTION ITEM TRACKER")
print("=" * 60)
print(df.to_string(index=False))

print(f"\nSummary:")
print(f"  Total items: {len(df)}")
print(f"  By priority: {df['priority'].value_counts().to_dict()}")
print(f"  By owner: {df['owner'].value_counts().to_dict()}")

## What You Learned
- **Transcript → structured notes** extraction
- **Action item extraction** with owners and deadlines
- **Follow-up email** generation
- **Meeting intelligence tracking** across meetings